# 06 - Artist Entity Matching

## Goal

Match Ticketmaster artist names to the correct MusicBrainz artist entities.

## Tasks

- Load MusicBrainz candidate matches
- Normalize artist names
- Identify exact name matches
- Automatically resolve clear matches
- Flag ambiguous matches for review
- Preserve unresolved artists
- Save the final artist mapping

In [4]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

In [ ]:
project_path = Path("..")

candidate_file = (
    project_path / "data" / "processed" / "musicbrainz" / "artist_candidates.csv"
)

candidates_df = pd.read_csv(candidate_file)

In [9]:
candidates_df.head()
candidates_df.shape

(833, 9)

## Artist Name Normalization

Artist names are normalized before comparison so that differences in capitalization, Unicode representation, and extra whitespace do not create false mismatches.

In [10]:
def normalize_artist_name(name):
    if pd.isna(name):
        return None
    
    name = unicodedata.normalize("NFKC", str(name))
    name = name.casefold()
    name = re.sub(r"\s+", " ", name).strip()

    return name 

In [12]:
candidates_df["tm_name_normalized"] = (
    candidates_df["ticketmaster_artist_name"].apply(normalize_artist_name)
)

candidates_df["mb_name_normalized"] = (
    candidates_df["musicbrainz_name"].apply(normalize_artist_name)
)

In [ ]:
candidates_df[["ticketmaster_artist_name",
        "musicbrainz_name",
        "tm_name_normalized",
        "mb_name_normalized"]]

,ticketmaster_artist_name,musicbrainz_name,tm_name_normalized,mb_name_normalized
0,54 Ultra,54 Ultra,54 ultra,54 ultra
1,6LACK,6LACK,6lack,6lack
2,8lanco,8lanco,8lanco,8lanco
3,A$AP Rocky,A$AP Rocky,a$ap rocky,a$ap rocky
4,Abbamania,ABBAmania Canada,abbamania,abbamania canada
...,...,...,...,...
828,Young Thug,Young T.H.U.G.,young thug,young t.h.u.g.
829,Yung Pepp,yung pepp,yung pepp,yung pepp
830,Zackavelli,Zackavelli,zackavelli,zackavelli
831,Zarna Garg,Zarna Garg,zarna garg,zarna garg


In [15]:
candidates_df["exact_name_match"] = (
    candidates_df["tm_name_normalized"]
    == candidates_df["mb_name_normalized"]
)

In [17]:
candidates_df.head()

,ticketmaster_artist_name,candidate_rank,mbid,musicbrainz_name,sort_name,artist_type,country,disambiguation,score,tm_name_normalized,mb_name_normalized,exact_name_match
0,54 Ultra,1,6f4926ae-865e-483b-86f9-9ae3b9584507,54 Ultra,54 Ultra,Person,NaN,NaN,100,54 ultra,54 ultra,True
1,6LACK,1,07832b42-8826-4ab1-acd3-c49a2f595ffe,6LACK,6LACK,Person,US,NaN,100,6lack,6lack,True
2,8lanco,1,641c819b-d57c-43c1-8bff-9e66f5fd2feb,8lanco,8lanco,Person,NO,NaN,100,8lanco,8lanco,True
3,A$AP Rocky,1,25b7b584-d952-4662-a8b9-dd8cdfbfeb64,A$AP Rocky,ASAP Rocky,Person,US,US rapper,100,a$ap rocky,a$ap rocky,True
4,Abbamania,1,bd2219b2-a506-4726-87e0-6406323b0d47,ABBAmania Canada,ABBAmania Canada,Group,CA,"Canadian ABBA tribute, not to be confused w/UK...",100,abbamania,abbamania canada,False


In [19]:
candidates_df["exact_name_match"].value_counts()

exact_name_match
True     523
False    310
Name: count, dtype: int64

## Match Resolution

Candidate groups are summarized per Ticketmaster artist to distinguish clear matches from ambiguous or unresolved cases.

In [20]:
artist_summary = (
    candidates_df.groupby("ticketmaster_artist_name").agg(candidate_count= ("mbid", "count"),
                                                          exact_match_count= ("exact_name_match", "sum")).reset_index()
)

In [21]:
artist_summary.head()

,ticketmaster_artist_name,candidate_count,exact_match_count
0,54 Ultra,1,1
1,6LACK,1,1
2,8lanco,1,1
3,A$AP Rocky,1,1
4,Abbamania,1,0


In [22]:
def classify_match(row):
    if row["exact_match_count"] == 1:
        return "auto_match"
    if row["exact_match_count"] > 1:
        return "ambiguous"
    return "unresolved"

In [23]:
artist_summary["match_status"] = (
    artist_summary.apply(classify_match, axis= 1)
)

In [25]:
artist_summary["match_status"].value_counts()

match_status
auto_match    293
ambiguous      82
unresolved     25
Name: count, dtype: int64

In [26]:
exact_candidates = candidates_df[candidates_df["exact_name_match"]].copy()

In [27]:
auto_match_names = set(
    artist_summary.loc[
        artist_summary["match_status"] == "auto_match",
        "ticketmaster_artist_name"
    ]
)

In [28]:
auto_matches = (
    exact_candidates[
        exact_candidates["ticketmaster_artist_name"]
        .isin(auto_match_names)
    ]
    .copy()
)

In [29]:
auto_matches["ticketmaster_artist_name"].is_unique

True

In [30]:
ambiguous_names = set(
    artist_summary.loc[
        artist_summary["match_status"] == "ambiguous",
        "ticketmaster_artist_name"
    ]
)

In [31]:
ambiguous_candidates = (
    candidates_df[
        candidates_df["ticketmaster_artist_name"]
        .isin(ambiguous_names)
    ]
    .sort_values(
        ["ticketmaster_artist_name", "candidate_rank"]
    )
)

In [32]:
ambiguous_candidates[
    [
        "ticketmaster_artist_name",
        "candidate_rank",
        "musicbrainz_name",
        "artist_type",
        "country",
        "score",
        "disambiguation",
        "mbid"
    ]
].head(30)

,ticketmaster_artist_name,candidate_rank,musicbrainz_name,artist_type,country,score,disambiguation,mbid
10,Alex Spencer,1,Alex Spencer,Person,NaN,100,Singer-songwriter from Manchester,09b55889-42af-4786-8eec-271bd5286c50
11,Alex Spencer,2,Alex Spencer,Person,US,97,NaN,5b08a1b6-021d-4842-9b94-3e3acab9471b
14,Amble,1,Amble,Group,IE,100,Trio of Irish contemporary folk musicians,c77f4f7a-da77-4487-aa29-a91cd4d29808
15,Amble,2,Amble,NaN,NaN,95,electronic music,9db476ca-78db-4951-8baf-6b972ca7fdaa
16,Amble,3,amble,Person,NaN,93,"Pianist, composer, improviser, Melbourne, Aust...",e022bece-7dfd-4e5d-87cc-6756a5d77544
17,Amble,4,Amble Skuse,Person,NaN,87,NaN,edabb19b-06ec-4e8d-aa56-88a0f9153fe4
18,Amble,5,Carl August Amble Eriksen,Person,NaN,75,NaN,8f49c162-9096-49d9-8046-ea77009f8fd7
22,Anastacia,1,Anastacia,Person,US,100,"USA pop singer from Chicago, IL, best known fo...",d3b2bec4-b70e-460e-b433-a865ceac2de8
23,Anastacia,2,Anastácia,Person,BR,87,"artist name of Lucinete Ferreira, Brazilian si...",aee2d8dc-5a00-4ffa-ae89-b120e57261b9
24,Anastacia,3,Anastacia,Person,RU,75,Russian singer,2f94a27d-129d-4078-9fc4-1fa3fd5c3942


In [33]:
unresolved_names = set(
    artist_summary.loc[
        artist_summary["match_status"] == "unresolved",
        "ticketmaster_artist_name"
    ]
)

In [ ]:
unresolved_candidates = (
    candidates_df[
        candidates_df["ticketmaster_artist_name"]
        .isin(unresolved_names)
    ]
    .sort_values(
        ["ticketmaster_artist_name", "candidate_rank"]
    )
)

In [35]:
artist_summary["match_status"].value_counts()

match_status
auto_match    293
ambiguous      82
unresolved     25
Name: count, dtype: int64

In [36]:
print("Candidate artists:", len(artist_summary))

print(
    "Auto matches:",
    (artist_summary["match_status"] == "auto_match").sum()
)

print(
    "Ambiguous:",
    (artist_summary["match_status"] == "ambiguous").sum()
)

print(
    "Unresolved:",
    (artist_summary["match_status"] == "unresolved").sum()
)

Candidate artists: 400
Auto matches: 293
Ambiguous: 82
Unresolved: 25


In [38]:
auto_matches[
    [
        "ticketmaster_artist_name",
        "musicbrainz_name",
        "mbid",
        "artist_type",
        "country"
    ]
].head(20)

,ticketmaster_artist_name,musicbrainz_name,mbid,artist_type,country
0,54 Ultra,54 Ultra,6f4926ae-865e-483b-86f9-9ae3b9584507,Person,NaN
1,6LACK,6LACK,07832b42-8826-4ab1-acd3-c49a2f595ffe,Person,US
2,8lanco,8lanco,641c819b-d57c-43c1-8bff-9e66f5fd2feb,Person,NO
3,A$AP Rocky,A$AP Rocky,25b7b584-d952-4662-a8b9-dd8cdfbfeb64,Person,US
5,aespa,aespa,b51c672b-85e0-48fe-8648-470a2422229f,Group,KR
6,Against Evil,Against Evil,33355567-84cf-423c-8add-2424949e1c08,Group,NaN
8,Aleks Syntek,Aleks Syntek,2fae3969-96cb-4dea-91c8-9108848de808,Person,MX
12,Allie Sherlock,Allie Sherlock,033f413f-325f-43db-aeaa-89b0d77299cd,Person,IE
13,Altered Rebirth,Altered Rebirth,0b6fb7e4-1e72-4291-ad3f-a2a0a0aa1b1b,Group,DE
19,Amelie Lens,Amelie Lens,67db280d-c3e4-49d9-978e-4050f4e209ef,Person,BE


In [39]:
artist_summary["match_status"].value_counts()

match_status
auto_match    293
ambiguous      82
unresolved     25
Name: count, dtype: int64

In [40]:
auto_matches["ticketmaster_artist_name"].is_unique

True

## Complete Artist Coverage

Artists without MusicBrainz candidates are added to the entity-matching summary so that every Ticketmaster artist is represented in the final mapping.

In [41]:
events_file = (
    project_path
    / "data"
    / "processed"
    / "ticketmaster_events_clean.csv"
)

events_df = pd.read_csv(events_file)

In [42]:
all_artists = (
    events_df[["artist_name"]]
    .dropna()
    .drop_duplicates()
    .rename(
        columns={
            "artist_name": "ticketmaster_artist_name"
        }
    )
    .reset_index(drop=True)
)

In [43]:
len(all_artists)

472

In [44]:
full_artist_summary = all_artists.merge(
    artist_summary,
    on="ticketmaster_artist_name",
    how="left"
)

In [45]:
full_artist_summary["match_status"] = (
    full_artist_summary["match_status"]
    .fillna("no_candidate")
)

In [46]:
full_artist_summary[
    ["candidate_count", "exact_match_count"]
] = (
    full_artist_summary[
        ["candidate_count", "exact_match_count"]
    ]
    .fillna(0)
    .astype(int)
)

In [47]:
full_artist_summary["match_status"].value_counts()

match_status
auto_match      293
ambiguous        82
no_candidate     72
unresolved       25
Name: count, dtype: int64

In [48]:
len(full_artist_summary)

472

In [49]:
selected_auto_matches = auto_matches[
    [
        "ticketmaster_artist_name",
        "mbid",
        "musicbrainz_name",
        "artist_type",
        "country",
        "disambiguation",
        "score"
    ]
].copy()

In [50]:
artist_mapping = full_artist_summary.merge(
    selected_auto_matches,
    on="ticketmaster_artist_name",
    how="left"
)

In [51]:
artist_mapping.head(20)

,ticketmaster_artist_name,candidate_count,exact_match_count,match_status,mbid,musicbrainz_name,artist_type,country,disambiguation,score
0,Heavysaurus,1,1,auto_match,430a7a48-14ba-4be0-8226-7340f9b3bbaf,Heavysaurus,Group,DE,German version of the Finnish band Heavisaurus,100.0
1,Melanie Martinez,3,3,ambiguous,NaN,NaN,NaN,NaN,NaN,NaN
2,SUPERBLOOM Festival,0,0,no_candidate,NaN,NaN,NaN,NaN,NaN,NaN
3,ITZY,3,2,ambiguous,NaN,NaN,NaN,NaN,NaN,NaN
4,Rawayana,1,1,auto_match,7de398f0-b4cf-476d-ac60-41d47d3511ca,Rawayana,Group,VE,NaN,100.0
5,Dance Gavin Dance,1,1,auto_match,16456fed-c9f2-4adf-b6ea-97b648c474d2,Dance Gavin Dance,Group,US,NaN,100.0
6,Jacob Collier,1,1,auto_match,1df15ee0-b52b-4315-9cb9-bc5a27a685e9,Jacob Collier,Person,GB,NaN,100.0
7,The Neighbourhood,5,2,ambiguous,NaN,NaN,NaN,NaN,NaN,NaN
8,Beartooth,1,1,auto_match,98a1e0ab-35fa-40dd-b62c-9fda46fdb061,Beartooth,Group,US,NaN,100.0
9,Hudson Freeman,1,1,auto_match,34356aec-8c4b-4f15-997c-e972cdede64d,Hudson Freeman,Person,NaN,NaN,100.0


In [52]:
artist_mapping.groupby(
    "match_status"
)["mbid"].apply(
    lambda x: x.notna().sum()
)

match_status
ambiguous         0
auto_match      293
no_candidate      0
unresolved        0
Name: mbid, dtype: int64

In [53]:
artist_mapping[
    artist_mapping["match_status"] == "auto_match"
]["mbid"].isna().sum()

np.int64(0)

## Secondary Matching for Unresolved Artists

Unresolved candidates are reviewed using normalized name similarity.

Similarity is used only as an additional signal and does not override ambiguous exact-name matches.

In [54]:
from difflib import SequenceMatcher

In [55]:
def name_similarity(name_a, name_b):
    if pd.isna(name_a) or pd.isna(name_b):
        return 0.0

    return SequenceMatcher(
        None,
        normalize_artist_name(name_a),
        normalize_artist_name(name_b)
    ).ratio()

In [56]:
unresolved_candidates = unresolved_candidates.copy()

unresolved_candidates["name_similarity"] = (
    unresolved_candidates.apply(
        lambda row: name_similarity(
            row["ticketmaster_artist_name"],
            row["musicbrainz_name"]
        ),
        axis=1
    )
)

In [57]:
unresolved_candidates[
    [
        "ticketmaster_artist_name",
        "candidate_rank",
        "musicbrainz_name",
        "score",
        "name_similarity",
        "artist_type",
        "country",
        "disambiguation"
    ]
].sort_values(
    ["ticketmaster_artist_name", "name_similarity"],
    ascending=[True, False]
).head(50)

,ticketmaster_artist_name,candidate_rank,musicbrainz_name,score,name_similarity,artist_type,country,disambiguation
4,Abbamania,1,ABBAmania Canada,100,0.720000,Group,CA,"Canadian ABBA tribute, not to be confused w/UK..."
38,Asgeir,1,Ásgeir,100,0.833333,Person,NaN,Icelandic singer-songwriter and musician
41,Asgeir,4,Asgeir Aarøen,75,0.631579,Person,NO,NaN
39,Asgeir,2,Asgeir Borgemoen,82,0.545455,Person,NO,NaN
42,Asgeir,5,Asgeir Søfteland,75,0.545455,Person,NO,NaN
40,Asgeir,3,Ásgeir Beinteinsson,76,0.400000,Person,IS,classical pianist
86,BENNETT,1,Tony Bennett,100,0.736842,Person,US,US jazz/standards vocalist
89,BENNETT,4,Alan Bennett,77,0.736842,Person,GB,British author
88,BENNETT,3,Brian Bennett,87,0.700000,Person,GB,Shadows drummer/library music composer
87,BENNETT,2,Warren Bennett,88,0.666667,Person,GB,NaN


In [58]:
best_unresolved_candidates = (
    unresolved_candidates
    .sort_values(
        [
            "ticketmaster_artist_name",
            "name_similarity",
            "score"
        ],
        ascending=[True, False, False]
    )
    .drop_duplicates(
        subset="ticketmaster_artist_name",
        keep="first"
    )
    .reset_index(drop=True)
)

In [59]:
best_unresolved_candidates["name_similarity"].describe()

count    25.000000
mean      0.822387
std       0.138433
min       0.500000
25%       0.736842
50%       0.875000
75%       0.923077
max       0.977778
Name: name_similarity, dtype: float64

In [60]:
best_unresolved_candidates[
    [
        "ticketmaster_artist_name",
        "musicbrainz_name",
        "name_similarity",
        "score",
        "disambiguation"
    ]
].sort_values(
    "name_similarity",
    ascending=False
).head(25)

,ticketmaster_artist_name,musicbrainz_name,name_similarity,score,disambiguation
23,"Yes, I'm Very Tired Now",Yes I'm Very Tired Now,0.977778,100,NaN
3,Bell Book & Candle,"Bell, Book & Candle",0.972973,100,"Jana Groß, Andy Birr & Hendrik Röder"
18,Stephen Wilson Jr.,Stephen Wilson Jr,0.971429,100,Singer songwriter
20,The Swingin’ Hermlins,The Swingin' Hermlins,0.952381,100,NaN
21,Torsten Strater,Torsten Sträter,0.933333,100,German comedian
14,Pi'erre Bourne,Pi’erre Bourne,0.928571,100,NaN
17,Skeler,skeler.,0.923077,100,NaN
12,Mother's Cake,Mother’s Cake,0.923077,100,Austrian psychedelic rock band
11,Marten Horger,Marten Hørger,0.923077,100,NaN
6,Gotz Widmann,Götz Widmann,0.916667,100,NaN


In [61]:
best_unresolved_candidates["name_similarity"].describe()

count    25.000000
mean      0.822387
std       0.138433
min       0.500000
25%       0.736842
50%       0.875000
75%       0.923077
max       0.977778
Name: name_similarity, dtype: float64

### Loose Name Normalization

A second normalization pass removes accents and punctuation to resolve formatting differences without relying only on fuzzy similarity.

In [62]:
def normalize_artist_name_loose(name):
    if pd.isna(name):
        return None

    name = unicodedata.normalize("NFKD", str(name))

    name = "".join(
        character
        for character in name
        if not unicodedata.combining(character)
    )

    name = name.casefold()

    name = re.sub(
        r"[^\w\s]",
        " ",
        name
    )

    name = re.sub(
        r"\s+",
        " ",
        name
    ).strip()

    return name

In [63]:
unresolved_candidates["tm_name_loose"] = (
    unresolved_candidates[
        "ticketmaster_artist_name"
    ].apply(normalize_artist_name_loose)
)

unresolved_candidates["mb_name_loose"] = (
    unresolved_candidates[
        "musicbrainz_name"
    ].apply(normalize_artist_name_loose)
)

In [64]:
unresolved_candidates["loose_exact_match"] = (
    unresolved_candidates["tm_name_loose"]
    == unresolved_candidates["mb_name_loose"]
)

In [65]:
unresolved_candidates[
    unresolved_candidates["loose_exact_match"]
][
    [
        "ticketmaster_artist_name",
        "musicbrainz_name",
        "candidate_rank",
        "score",
        "name_similarity"
    ]
]

,ticketmaster_artist_name,musicbrainz_name,candidate_rank,score,name_similarity
38,Asgeir,Ásgeir,1,100,0.833333
76,Bell Book & Candle,"Bell, Book & Candle",1,100,0.972973
77,Bell Book & Candle,"Bell, Book & Candle",2,87,0.972973
152,Die Arzte,Die Ärzte,1,100,0.888889
184,Ed O'brien,Ed O’Brien,1,100,0.900000
185,Ed O'brien,Ed O’Brien,2,94,0.900000
249,Gotz Widmann,Götz Widmann,1,100,0.916667
362,Kaye Ree,Kaye-Ree,1,100,0.875000
493,Mother's Cake,Mother’s Cake,1,100,0.923077
565,Pi'erre Bourne,Pi’erre Bourne,1,100,0.928571


In [66]:
loose_match_summary = (
    unresolved_candidates
    .groupby("ticketmaster_artist_name")
    ["loose_exact_match"]
    .sum()
)

loose_match_summary.value_counts()

loose_exact_match
1    12
0    10
2     3
Name: count, dtype: int64

In [67]:
secondary_match_names = set(
    loose_match_summary[
        loose_match_summary == 1
    ].index
)

In [68]:
secondary_matches = (
    unresolved_candidates[
        unresolved_candidates[
            "ticketmaster_artist_name"
        ].isin(secondary_match_names)
        &
        unresolved_candidates[
            "loose_exact_match"
        ]
    ]
    .copy()
)

In [69]:
secondary_matches[
    "ticketmaster_artist_name"
].is_unique

True

In [70]:
secondary_matches[
    [
        "ticketmaster_artist_name",
        "musicbrainz_name",
        "mbid",
        "name_similarity",
        "disambiguation"
    ]
]

,ticketmaster_artist_name,musicbrainz_name,mbid,name_similarity,disambiguation
38,Asgeir,Ásgeir,380429da-3827-43fd-9e67-558e4c6a91bf,0.833333,Icelandic singer-songwriter and musician
152,Die Arzte,Die Ärzte,f2fb0ff0-5679-42ec-a55c-15109ce6e320,0.888889,NaN
249,Gotz Widmann,Götz Widmann,bb0085ff-c63b-4611-b582-40b7797d4a04,0.916667,NaN
362,Kaye Ree,Kaye-Ree,11e803cb-069f-409d-b3c7-5f3d575b1b75,0.875000,NaN
493,Mother's Cake,Mother’s Cake,31ea4b40-f773-402c-aa97-a35b5aa825e4,0.923077,Austrian psychedelic rock band
565,Pi'erre Bourne,Pi’erre Bourne,be0d7749-0974-48ba-93bb-6e974ec193a2,0.928571,NaN
587,Pro Pain,Pro‐Pain,ee94cb90-504b-414d-9d09-1250d7d6192e,0.875000,NaN
679,Skeler,skeler.,4cb4ebd9-3a15-406b-aadd-b5401754ba40,0.923077,NaN
705,Stephen Wilson Jr.,Stephen Wilson Jr,c28bcdb2-6e32-4b15-a9b0-77a4d88c6c74,0.971429,Singer songwriter
746,The Swingin’ Hermlins,The Swingin' Hermlins,9851c51d-f277-4f2e-9589-197bce924dbe,0.952381,NaN


## Apply Secondary Matches

Unique loose-name matches are accepted as secondary matches after review of formatting and punctuation differences.

In [72]:
selected_secondary_matches = (
    secondary_matches
    .rename(
        columns={
            "mbid": "secondary_mbid",
            "musicbrainz_name": "secondary_musicbrainz_name",
            "artist_type": "secondary_artist_type",
            "country": "secondary_country",
            "disambiguation": "secondary_disambiguation",
            "score": "secondary_score"
        }
    )
)

In [73]:
artist_mapping = artist_mapping.merge(
    selected_secondary_matches,
    on="ticketmaster_artist_name",
    how="left"
)

In [74]:
artist_mapping["secondary_mbid"].notna().sum()

np.int64(12)

In [75]:
secondary_mask = (
    artist_mapping["secondary_mbid"].notna()
)

In [76]:
artist_mapping.loc[
    secondary_mask,
    "mbid"
] = artist_mapping.loc[
    secondary_mask,
    "secondary_mbid"
]

In [77]:
artist_mapping.loc[
    secondary_mask,
    "musicbrainz_name"
] = artist_mapping.loc[
    secondary_mask,
    "secondary_musicbrainz_name"
]

artist_mapping.loc[
    secondary_mask,
    "artist_type"
] = artist_mapping.loc[
    secondary_mask,
    "secondary_artist_type"
]

artist_mapping.loc[
    secondary_mask,
    "country"
] = artist_mapping.loc[
    secondary_mask,
    "secondary_country"
]

artist_mapping.loc[
    secondary_mask,
    "disambiguation"
] = artist_mapping.loc[
    secondary_mask,
    "secondary_disambiguation"
]

artist_mapping.loc[
    secondary_mask,
    "score"
] = artist_mapping.loc[
    secondary_mask,
    "secondary_score"
]

In [78]:
artist_mapping.loc[
    secondary_mask,
    "match_status"
] = "secondary_match"

In [79]:
artist_mapping["match_status"].value_counts()

match_status
auto_match         293
ambiguous           82
no_candidate        72
unresolved          13
secondary_match     12
Name: count, dtype: int64

In [80]:
resolved_mask = artist_mapping["match_status"].isin(
    ["auto_match", "secondary_match"]
)

print("Resolved artists:", resolved_mask.sum())

print(
    "Resolved artists missing MBID:",
    artist_mapping.loc[
        resolved_mask,
        "mbid"
    ].isna().sum()
)

Resolved artists: 305
Resolved artists missing MBID: 0


In [81]:
unresolved_mask = ~resolved_mask

print(
    "Unresolved artists with MBID:",
    artist_mapping.loc[
        unresolved_mask,
        "mbid"
    ].notna().sum()
)

Unresolved artists with MBID: 0


In [82]:
artist_mapping = artist_mapping.drop(
    columns=[
        "secondary_mbid",
        "secondary_musicbrainz_name",
        "secondary_artist_type",
        "secondary_country",
        "secondary_disambiguation",
        "secondary_score"
    ]
)

In [83]:
artist_mapping.shape

(472, 19)

In [84]:
print(
    "Total artists:",
    len(artist_mapping)
)

print(
    "Unique Ticketmaster artists:",
    artist_mapping["ticketmaster_artist_name"].is_unique
)

print(
    "Duplicate Ticketmaster artists:",
    artist_mapping["ticketmaster_artist_name"]
    .duplicated()
    .sum()
)

Total artists: 472
Unique Ticketmaster artists: True
Duplicate Ticketmaster artists: 0


In [85]:
status_counts = artist_mapping["match_status"].value_counts()

resolved_mask = artist_mapping["match_status"].isin(
    ["auto_match", "secondary_match"]
)

print("Total artists:", len(artist_mapping))
print(
    "Unique Ticketmaster artists:",
    artist_mapping["ticketmaster_artist_name"].is_unique
)
print(
    "Duplicate Ticketmaster artists:",
    artist_mapping["ticketmaster_artist_name"]
    .duplicated()
    .sum()
)

print("\nMatch status:")
print(status_counts)

print(
    "\nResolved artists:",
    resolved_mask.sum()
)

print(
    "Resolved artists missing MBID:",
    artist_mapping.loc[
        resolved_mask,
        "mbid"
    ].isna().sum()
)

print(
    "Unresolved artists with MBID:",
    artist_mapping.loc[
        ~resolved_mask,
        "mbid"
    ].notna().sum()
)

Total artists: 472
Unique Ticketmaster artists: True
Duplicate Ticketmaster artists: 0

Match status:
match_status
auto_match         293
ambiguous           82
no_candidate        72
unresolved          13
secondary_match     12
Name: count, dtype: int64

Resolved artists: 305
Resolved artists missing MBID: 0
Unresolved artists with MBID: 0


In [86]:
resolution_rate = (
    resolved_mask.sum()
    / len(artist_mapping)
    * 100
)

print(
    f"Resolution rate: {resolution_rate:.1f}%"
)

Resolution rate: 64.6%


## Save Final Artist Mapping

The final artist mapping preserves confident MusicBrainz identities while leaving ambiguous, unresolved, and unmatched artists without forced assignments.

In [87]:
mapping_file = (
    project_path
    / "data"
    / "processed"
    / "musicbrainz"
    / "artist_mapping.csv"
)

artist_mapping.to_csv(
    mapping_file,
    index=False
)

print("Mapping saved:", mapping_file.exists())
print("Saved rows:", len(artist_mapping))

Mapping saved: True
Saved rows: 472


In [88]:
ambiguous_file = (
    project_path
    / "data"
    / "processed"
    / "musicbrainz"
    / "ambiguous_artist_candidates.csv"
)

ambiguous_candidates.to_csv(
    ambiguous_file,
    index=False
)

In [89]:
remaining_unresolved_names = set(
    artist_mapping.loc[
        artist_mapping["match_status"] == "unresolved",
        "ticketmaster_artist_name"
    ]
)

remaining_unresolved_candidates = candidates_df[
    candidates_df["ticketmaster_artist_name"]
    .isin(remaining_unresolved_names)
].copy()

unresolved_file = (
    project_path
    / "data"
    / "processed"
    / "musicbrainz"
    / "unresolved_artist_candidates.csv"
)

remaining_unresolved_candidates.to_csv(
    unresolved_file,
    index=False
)

## Findings

- All Ticketmaster artists were preserved in the final mapping.
- Clear normalized-name matches were automatically resolved.
- Additional formatting, punctuation, and accent differences were resolved through loose normalization.
- Ambiguous artists were not force-matched when multiple plausible MusicBrainz identities existed.
- Unresolved and no-candidate artists were preserved without assigning unreliable MBIDs.
- The resulting mapping provides a conservative and explainable foundation for downstream artist enrichment and recommendations.